# Numpy notebook 14/10/2026
### Brais Otero Lema

Exercise 1. Neighbors.

In [3]:
import numpy as np
from scipy.special import gamma

rng = np.random.default_rng(2664)

L, N = 10., 1_000

r = rng.uniform(0, L, (N, 3))

def dist_box(r, L, pbc=True):
    dr = r[np.newaxis, :] - r[:, np.newaxis]
    if pbc:
        dr -= L * np.round(dr/L)
    ds = np.sqrt(np.sum(dr**2, axis=-1))
    return ds

ds = dist_box(r, L)

print("Checking:")
print(f"ds shape: {ds.shape}")
print(f"ds symmetric: {np.all(ds == ds.T)}")
print(f"no distance over sqrt(3)/2 L: {np.all(ds < np.sqrt(3)/2*L)}")

ds_nopbc = dist_box(r, L, pbc=False)
strs = ("with", "without")

expected_mean_nnd = gamma(4/3) * np.power(4*np.pi/3*N/L**3, -1/3)
print(f"\nExpected mean nearest neighbor distance: {expected_mean_nnd}")
for d, st in zip((ds, ds_nopbc), strs):
    nearest_neighbor_distance = np.sort(d, axis=0)[1]
    mean_nnd = np.mean(nearest_neighbor_distance)
    dev_nnd = np.power(N*(N-1), -1/2) * np.sqrt(np.sum((nearest_neighbor_distance - mean_nnd)**2))
    within_error = mean_nnd - dev_nnd <= expected_mean_nnd <= mean_nnd + dev_nnd

    print(f"Computing {st} pbc:")
    print(f"- Mean nearest neighbor distance {st} pbc: {mean_nnd}")
    print(f"- Statistical error {st} pbc: {dev_nnd}")
    print(f"- Is expected within error: {within_error}")


Checking:
ds shape: (1000, 1000)
ds symmetric: True
no distance over sqrt(3)/2 L: True

Expected mean nearest neighbor distance: 0.5539602783650901
Computing with pbc:
- Mean nearest neighbor distance with pbc: 0.5595830532308397
- Statistical error with pbc: 0.006285602221567857
- Is expected within error: True
Computing without pbc:
- Mean nearest neighbor distance without pbc: 0.580561052432909
- Statistical error without pbc: 0.006969637131449798
- Is expected within error: False


Once periodic boundary conditions are removed, the gas is no longer constrained to the volume of the box, and so the mean distance between nearest neighbors increases slightly due to an expansion of the gas. The effect is noticeable enough that the result is no longer compatible with statistical mechanics predictions, which have been done precisely in the canonical ensemble, where system volume is fixed.

Exercise 2. Problematic run.

In [4]:
import numpy as np

rng = np.random.default_rng(2664)

n = 6_000

time = rng.uniform(0, 120, n)
chan = rng.integers(0, 12, n)
energy = rng.exponential(1.5, n)

# Detector workings
hot = rng.random(n) < .08
chan[hot] = 7
energy[hot] = rng.exponential(.2, hot.sum())
off = (time > 61.3) & (time < 68.9)
time, chan, energy = time[~off], chan[~off], energy[~off]
energy[rng.random(time.size) < .03] = np.nan

time_order = np.argsort(time)
time, chan, energy = time[time_order], chan[time_order], energy[time_order]

failed_readouts = np.isnan(energy)
time, chan, energy = time[~failed_readouts], chan[~failed_readouts], energy[~failed_readouts]
print(f"Number of failed readouts: {sum(failed_readouts)}\n")

events_per_channel = np.bincount(chan)
mean_energy_per_channel = np.bincount(chan, weights=energy) / events_per_channel
for channel, (events, mean_energy) in enumerate(zip(events_per_channel, mean_energy_per_channel)):
    print(f"Channel {channel:02} with {events} events has mean energy: {mean_energy}")

time_cutoff = np.where(np.diff(time) > 1)[0][0]
print(f"\nDetector turned off from t = {time[time_cutoff]:.2f} s to t = {time[time_cutoff+1]:.2f} s")


Number of failed readouts: 170

Channel 00 with 425 events has mean energy: 1.468363599823443
Channel 01 with 429 events has mean energy: 1.428234008195584
Channel 02 with 414 events has mean energy: 1.524716891585129
Channel 03 with 414 events has mean energy: 1.5898247316801546
Channel 04 with 414 events has mean energy: 1.5137214662047194
Channel 05 with 410 events has mean energy: 1.4237069069813728
Channel 06 with 427 events has mean energy: 1.5290623279854585
Channel 07 with 879 events has mean energy: 0.8207617033508476
Channel 08 with 403 events has mean energy: 1.4138020179606585
Channel 09 with 414 events has mean energy: 1.456683557906186
Channel 10 with 437 events has mean energy: 1.5549693734664072
Channel 11 with 367 events has mean energy: 1.493905031313824

Detector turned off from t = 61.27 s to t = 68.92 s


The odd channel out is the eight one (index 07). It has about twice as many events as the rest of the channels because the detector's inner workings incorrectly assign other channels' events to this one, with a random probability. This in turn contributes an increase in the number of events attributed to this channel while the energy does not increase in the same manner, which leads to the strange mean energy observed.